# SC-3-Solidity-Basics - Fondements de Solidity

[<< Setup Web3py](../00-Foundations/SC-2-Setup-Web3py.ipynb) | [Functions & State >>](SC-4-Functions-State.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre la structure d'un contrat Solidity
2. Maitriser les **types de donnees** (value types)
3. Utiliser les **variables** d'etat et locales
4. Effectuer des **conversions** de types

### Prerequis

- Notions de base en programmation (variables, fonctions, boucles)
- Environnement configure via [SC-0-Setup](../00-Foundations/SC-0-Setup.ipynb)
- Node.js installe (pour Hardhat optionnel)

### Duree estimee : 40 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


Connecte a anvil (chain 31337), deployer: 0xf39Fd6e5...


---

## 1. Structure d'un contrat

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract MonContrat {
    // Variables d'etat
    // Fonctions
}
```

### Elements

| Element | Description |
|---------|-------------|
| `pragma` | Version du compilateur |
| `contract` | Mot-cle de definition |
| `{ }` | Corps du contrat |

In [2]:
# Exemple de contrat minimal
CONTRACT_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Counter {
    uint256 public count;

    function increment() public {
        count += 1;
    }

    function getCount() public view returns (uint256) {
        return count;
    }
}
'''


# Compilation et deploiement reel sur anvil
counter, receipt = compile_and_deploy(w3, CONTRACT_EXAMPLE, deployer)

Deploye: Counter a 0x5FbDB2315678afecb367f032d93F642f64180aa3


---

## 2. Types de donnees (Value Types)

### 2.1 Booleens

In [3]:
# Types booleens - compilation et deploiement reel
BOOL_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract BoolExample {
    bool public isActive = true;
    bool public isPaused = false;

    function toggle() public {
        isActive = !isActive;
    }

    function logicalAnd(bool a, bool b) public pure returns (bool) {
        return a && b;
    }

    function logicalOr(bool a, bool b) public pure returns (bool) {
        return a || b;
    }

    function logicalNot(bool a) public pure returns (bool) {
        return !a;
    }
}
'''

print("--- Booleens: bool (true/false), && (and), || (or), ! (not) ---")
bool_contract, receipt = compile_and_deploy(w3, BOOL_EXAMPLE, deployer)
print(f"  Deploye : {bool_contract.address}")
print(f"  isActive = {bool_contract.functions.isActive().call()}")
bool_contract.functions.toggle().transact({'from': deployer})
print(f"  Apres toggle() : {bool_contract.functions.isActive().call()}")
print(f"  True && False = {bool_contract.functions.logicalAnd(True, False).call()}")
print(f"  True || False = {bool_contract.functions.logicalOr(True, False).call()}")

--- Booleens: bool (true/false), && (and), || (or), ! (not) ---


Deploye: BoolExample a 0xDc64a140Aa3E981100a9becA4E685f962f0cF6C9
  Deploye : 0xDc64a140Aa3E981100a9becA4E685f962f0cF6C9
  isActive = True
  Apres toggle() : False
  True && False = False
  True || False = True


### 2.2 Entiers

In [4]:
# Types entiers - compilation et deploiement reel
INT_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract IntExample {
    // Entiers non signes (positifs)
    uint8 public smallNumber;      // 0 a 255
    uint256 public bigNumber;      // 0 a 2^256-1
    uint public defaultUint;       // uint256 par defaut

    // Entiers signes (positifs et negatifs)
    int8 public signedSmall;       // -128 a 127
    int256 public signedBig;       // -2^255 a 2^255-1

    function add(uint a, uint b) public pure returns (uint) {
        return a + b;
    }

    function subtract(int a, int b) public pure returns (int) {
        return a - b;
    }

    function multiply(uint a, uint b) public pure returns (uint) {
        return a * b;
    }

    function modulo(uint a, uint b) public pure returns (uint) {
        return a % b;
    }
}
'''

print("--- Entiers: uint8, uint256, int8, int256 (signes/non signes) ---")
int_contract, receipt = compile_and_deploy(w3, INT_EXAMPLE, deployer)
print(f"  Deploye : {int_contract.address}")
print(f"  add(100, 200) = {int_contract.functions.add(100, 200).call()}")
print(f"  subtract(10, 3) = {int_contract.functions.subtract(10, 3).call()}")
print(f"  multiply(7, 6) = {int_contract.functions.multiply(7, 6).call()}")
print(f"  modulo(17, 5) = {int_contract.functions.modulo(17, 5).call()}")

--- Entiers: uint8, uint256, int8, int256 (signes/non signes) ---


Deploye: IntExample a 0xA51c1fc2f0D1a1b8494Ed1FE312d7C3a78Ed91C0
  Deploye : 0xA51c1fc2f0D1a1b8494Ed1FE312d7C3a78Ed91C0
  add(100, 200) = 300
  subtract(10, 3) = 7
  multiply(7, 6) = 42
  modulo(17, 5) = 2


### 2.3 Addresses

In [5]:
# Types adresse - compilation et deploiement reel
ADDRESS_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract AddressExample {
    address public owner;
    address payable public treasury;

    constructor() {
        owner = msg.sender;
    }

    function getBalance(address addr) public view returns (uint256) {
        return addr.balance;
    }

    function isContract(address addr) public view returns (bool) {
        uint256 size;
        assembly { size := extcodesize(addr) }
        return size > 0;
    }
}
'''

print("--- Adresses: address (20 octets), address payable (transfert ETH) ---")
addr_contract, receipt = compile_and_deploy(w3, ADDRESS_EXAMPLE, deployer)
print(f"  Deploye : {addr_contract.address}")
print(f"  owner = {addr_contract.functions.owner().call()}")
print(f"  deployer = {deployer}")
print(f"  owner == deployer : {addr_contract.functions.owner().call().lower() == deployer.lower()}")
balance = addr_contract.functions.getBalance(deployer).call()
print(f"  balance(deployer) = {w3.from_wei(balance, 'ether'):.2f} ETH")

--- Adresses: address (20 octets), address payable (transfert ETH) ---
Deploye: AddressExample a 0x959922bE3CAee4b8Cd9a407cc3ac1C251C2007B1
  Deploye : 0x959922bE3CAee4b8Cd9a407cc3ac1C251C2007B1
  owner = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
  deployer = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
  owner == deployer : True


  balance(deployer) = 10000.00 ETH


### 2.4 Bytes et String

In [6]:
# Types bytes et string - compilation et deploiement reel
BYTES_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract BytesExample {
    bytes32 public fixedData = "hello";
    bytes public dynamicData = "world";
    string public message = "Hello Solidity";

    function setFixed(bytes32 _data) public {
        fixedData = _data;
    }

    function setMessage(string memory _msg) public {
        message = _msg;
    }

    function getLength() public view returns (uint) {
        return bytes(message).length;
    }

    function concatenate(string memory a, string memory b) public pure returns (string memory) {
        return string(abi.encodePacked(a, b));
    }
}
'''

print("--- Bytes/String: bytes32 (fixe), bytes (dynamique), string (UTF-8) ---")
bytes_contract, receipt = compile_and_deploy(w3, BYTES_EXAMPLE, deployer)
print(f"  Deploye : {bytes_contract.address}")
print(f"  message = '{bytes_contract.functions.message().call()}'")
print(f"  getLength() = {bytes_contract.functions.getLength().call()}")
bytes_contract.functions.setMessage("Solidity rocks!").transact({'from': deployer})
print(f"  Apres setMessage : '{bytes_contract.functions.message().call()}'")
result = bytes_contract.functions.concatenate("Hello", " World").call()
print(f"  concatenate('Hello', ' World') = '{result}'")

--- Bytes/String: bytes32 (fixe), bytes (dynamique), string (UTF-8) ---


Deploye: BytesExample a 0x4A679253410272dd5232B3Ff7cF5dbB88f295319
  Deploye : 0x4A679253410272dd5232B3Ff7cF5dbB88f295319
  message = 'Hello Solidity'
  getLength() = 14
  Apres setMessage : 'Solidity rocks!'
  concatenate('Hello', ' World') = 'Hello World'


---

## 3. Variables

### 3.1 Variables d'etat

In [7]:
# Variables d'etat - compilation et deploiement reel
STATE_VARS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract StateVariables {
    // Stockees en permanence sur la blockchain
    uint256 public storedData = 42;
    address public owner;
    bool private initialized = false;

    // Visibilite:
    // public   - getter auto-genere, visible de partout
    // private  - uniquement ce contrat
    // internal - ce contrat + contrats enfants
    // external - uniquement depuis l'exterieur (fonctions seulement)

    constructor() {
        owner = msg.sender;
        initialized = true;
    }

    function setData(uint256 _data) public {
        storedData = _data;
    }

    function isInitialized() public view returns (bool) {
        return initialized;
    }
}
'''

print("--- Variables d'etat : persistees sur la blockchain entre les appels ---")
state_contract, receipt = compile_and_deploy(w3, STATE_VARS, deployer)
print(f"  Deploye : {state_contract.address}")
print(f"  storedData = {state_contract.functions.storedData().call()}")
print(f"  owner = {state_contract.functions.owner().call()}")
print(f"  isInitialized() = {state_contract.functions.isInitialized().call()}")
state_contract.functions.setData(100).transact({'from': deployer})
print(f"  Apres setData(100) : storedData = {state_contract.functions.storedData().call()}")

--- Variables d'etat : persistees sur la blockchain entre les appels ---


Deploye: StateVariables a 0x851356ae760d987E095750cCeb3bC6014560891C
  Deploye : 0x851356ae760d987E095750cCeb3bC6014560891C
  storedData = 42
  owner = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
  isInitialized() = True
  Apres setData(100) : storedData = 100


### 3.2 Variables locales

In [8]:
# Variables locales - compilation et deploiement reel
LOCAL_VARS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract LocalVariables {
    uint256 public result;

    function calculate(uint a, uint b) public returns (uint256) {
        // Variables locales (en memoire, non stockees sur la blockchain)
        uint256 sum = a + b;
        uint256 product = a * b;

        // Seul le resultat final est stocke
        result = sum + product;
        return result;
    }

    function computeOnly(uint a, uint b) public pure returns (uint256 sum, uint256 product) {
        // pure : ni lecture ni ecriture sur la blockchain
        sum = a + b;
        product = a * b;
    }
}
'''

print("--- Variables locales : temporaires, en memoire (non stockees) ---")
local_contract, receipt = compile_and_deploy(w3, LOCAL_VARS, deployer)
print(f"  Deploye : {local_contract.address}")
print(f"  calculate(3, 4) = {local_contract.functions.calculate(3, 4).call()}")
local_contract.functions.calculate(3, 4).transact({'from': deployer})
print(f"  result sur la blockchain = {local_contract.functions.result().call()}")
s, p = local_contract.functions.computeOnly(5, 6).call()
print(f"  computeOnly(5, 6) : sum={s}, product={p} (pas stocke)")

--- Variables locales : temporaires, en memoire (non stockees) ---
Deploye: LocalVariables a 0x0E801D84Fa97b50751Dbf25036d067dCf18858bF
  Deploye : 0x0E801D84Fa97b50751Dbf25036d067dCf18858bF
  calculate(3, 4) = 19
  result sur la blockchain = 19
  computeOnly(5, 6) : sum=11, product=30 (pas stocke)


### 3.3 Variables globales

In [9]:
# Variables globales - compilation et deploiement reel
GLOBAL_VARS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract GlobalVariables {
    event GlobalsRead(
        address sender,
        uint256 timestamp,
        uint256 blockNumber,
        uint256 gasLimit
    );

    function getGlobals() public returns (
        address sender,
        uint256 timestamp,
        uint256 blockNumber,
        uint256 gasLimit
    ) {
        sender = msg.sender;
        timestamp = block.timestamp;
        blockNumber = block.number;
        gasLimit = block.gaslimit;
        emit GlobalsRead(sender, timestamp, blockNumber, gasLimit);
    }

    function getMsgValue() public payable returns (uint256) {
        return msg.value;
    }
}
'''

print("--- Variables globales : msg.sender, block.timestamp, block.number ---")
globals_contract, receipt = compile_and_deploy(w3, GLOBAL_VARS, deployer)
print(f"  Deploye : {globals_contract.address}")
sender, ts, block_n, gas_lim = globals_contract.functions.getGlobals().call({'from': deployer})
print(f"  msg.sender   = {sender}")
print(f"  block.timestamp = {ts}")
print(f"  block.number = {block_n}")
print(f"  block.gaslimit = {gas_lim:,}")

--- Variables globales : msg.sender, block.timestamp, block.number ---


Deploye: GlobalVariables a 0x5f3f1dBD7B74C6B46e8c44f98792A1dAf8d69154
  Deploye : 0x5f3f1dBD7B74C6B46e8c44f98792A1dAf8d69154
  msg.sender   = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
  block.timestamp = 1776606445
  block.number = 53
  block.gaslimit = 30,000,000


---

## 4. Conversions de types

In [10]:
# Conversions
CONVERSION_EXAMPLE = '''
contract TypeConversions {
    // Conversion explicite
    function uintToInt(uint256 a) public pure returns (int256) {
        return int256(a);
    }

    // Conversion address to payable
    function toPayable(address addr) public pure returns (address payable) {
        return payable(addr);
    }

    // String to bytes
    function stringToBytes(string memory s) public pure returns (bytes memory) {
        return bytes(s);
    }

    // Address to uint (pour comparaison)
    function addressToUint(address addr) public pure returns (uint160) {
        return uint160(addr);
    }
}
'''

print("Conversions de types Solidity:")
print(CONVERSION_EXAMPLE)

Conversions de types Solidity:

contract TypeConversions {
    // Conversion explicite
    function uintToInt(uint256 a) public pure returns (int256) {
        return int256(a);
    }

    // Conversion address to payable
    function toPayable(address addr) public pure returns (address payable) {
        return payable(addr);
    }

    // String to bytes
    function stringToBytes(string memory s) public pure returns (bytes memory) {
        return bytes(s);
    }

    // Address to uint (pour comparaison)
    function addressToUint(address addr) public pure returns (uint160) {
        return uint160(addr);
    }
}



---

## 5. Exercices

### Exercice 1 : Contrat SimpleStorage

Creez un contrat qui stocke et recupere une valeur uint256.

In [11]:
# Votre code ici
EXERCICE_1 = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SimpleStorage {
    // TODO: Declarer une variable d'etat privee de type uint256

    function set(uint256 _value) public {
        // TODO: Stocker la valeur passee en parametre
    }

    function get() public view returns (uint256) {
        // TODO: Retourner la valeur stockee
    }
}
'''

# Compilation et deploiement reel sur anvil
simplestorage, receipt = compile_and_deploy(w3, EXERCICE_1, deployer)

Deploye: SimpleStorage a 0x2bdCC0de6bE1f7D2ee689a0342D76F52E8EFABa3


### Exercice 2 : Contrat Owner

Creez un contrat qui stocke l'adresse du proprietaire et permet de la changer.

In [12]:
# Votre code ici
EXERCICE_2 = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Owner {
    address public owner;

    constructor() {
        // TODO: Initialiser owner avec l'adresse du deployeur (msg.sender)
    }

    function changeOwner(address _newOwner) public {
        // TODO: Changer le proprietaire
    }
}
'''

# Compilation et deploiement reel sur anvil
owner, receipt = compile_and_deploy(w3, EXERCICE_2, deployer)

Deploye: Owner a 0xc351628EB244ec633d5f21fBD6621e1a683B1181


---

## 6. Resume

| Type | Description | Exemple |
|------|-------------|---------|
| `bool` | Booleen | `true`, `false` |
| `uint256` | Entier non signe | `42`, `1000000` |
| `int256` | Entier signe | `-42`, `42` |
| `address` | Adresse Ethereum | `0x1234...` |
| `bytes32` | Bytes fixes | `hex"abcd"` |
| `string` | Chaine UTF-8 | `"Hello"` |

---

**Notebook suivant** : [SC-4-Functions-State](SC-4-Functions-State.ipynb)

---

[<< Setup Web3py](../00-Foundations/SC-2-Setup-Web3py.ipynb) | [Functions & State >>](SC-4-Functions-State.ipynb)